# ParseIQ Performance Analysis

> **Author:** Shriniwas Ahirrao  
> **Project:** ParseIQ — AI-Powered Data Quality Agent  
> **Date:** April 2026

This notebook provides a comprehensive performance analysis of ParseIQ, including:
1. **Execution Time Benchmarks** — ParseIQ vs ydata-profiling vs Great Expectations
2. **Anomaly Detection Accuracy** — Precision, Recall, F1 on known-anomaly datasets
3. **Quality Score Correlation** — How ParseIQ scores compare to expert assessments
4. **Scalability Analysis** — Performance across dataset sizes (1K to 100K records)
5. **Memory Usage Profiling** — Peak memory consumption during analysis
6. **Nested JSON Flattening Performance** — Depth vs time characteristics
7. **LLM Enrichment Overhead** — Provider latency comparison

## Setup & Imports

In [ ]:
import json
import time
import os
import sys
import tracemalloc
from pathlib import Path

import numpy as np
import pandas as pd

# Visualisation
try:
    import matplotlib.pyplot as plt
    import matplotlib.ticker as mticker
    plt.style.use('seaborn-v0_8-darkgrid')
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
    print('matplotlib not installed — install with: pip install matplotlib')

# ParseIQ
sys.path.insert(0, str(Path('.').resolve().parent))
from parseiq import Pipeline, __version__
print(f'ParseIQ version: {__version__}')
print(f'Python version: {sys.version}')
print(f'NumPy version: {np.__version__}')
print(f'pandas version: {pd.__version__}')

## 1. Execution Time Benchmarks

Compare ParseIQ's execution time against other data profiling tools across different dataset sizes.

In [ ]:
def generate_test_data(n_records, n_columns=10, seed=42):
    """Generate a test DataFrame with mixed types for benchmarking."""
    rng = np.random.default_rng(seed)
    data = {}
    for i in range(n_columns):
        col_type = i % 4
        if col_type == 0:  # integer
            data[f'int_col_{i}'] = rng.integers(0, 1000, size=n_records)
        elif col_type == 1:  # float
            data[f'float_col_{i}'] = rng.normal(50, 15, size=n_records)
        elif col_type == 2:  # string
            choices = ['alpha', 'beta', 'gamma', 'delta', 'epsilon', None]
            data[f'str_col_{i}'] = rng.choice(choices, size=n_records)
        else:  # boolean
            data[f'bool_col_{i}'] = rng.choice([True, False, None], size=n_records)
    
    # Inject some anomalies
    data['int_col_0'][0] = -999  # negative outlier
    data['float_col_1'][1] = 999.99  # numeric outlier
    
    return pd.DataFrame(data)

# Dataset sizes to benchmark
SIZES = [100, 500, 1_000, 5_000, 10_000, 50_000, 100_000]

print('Test data generator ready.')
print(f'Will benchmark across sizes: {SIZES}')

In [ ]:
def benchmark_parseiq(df, output_dir='benchmark_output'):
    """Benchmark ParseIQ on a DataFrame."""
    os.makedirs(output_dir, exist_ok=True)
    # Save to temp JSON
    temp_path = os.path.join(output_dir, 'bench_data.json')
    records = df.astype(object).where(df.notna(), None).to_dict(orient='records')
    with open(temp_path, 'w') as f:
        json.dump(records, f, default=str)
    
    start = time.perf_counter()
    result = Pipeline(temp_path).run(llm=False, output_dir=output_dir, force=True)
    elapsed = time.perf_counter() - start
    
    os.remove(temp_path)
    return elapsed, result


def benchmark_ydata(df):
    """Benchmark ydata-profiling on a DataFrame."""
    try:
        from ydata_profiling import ProfileReport
    except ImportError:
        return None
    
    start = time.perf_counter()
    profile = ProfileReport(df, minimal=True, progress_bar=False)
    _ = profile.to_json()  # Force computation
    elapsed = time.perf_counter() - start
    return elapsed


print('Benchmark functions defined.')

In [ ]:
# Run benchmarks
parseiq_times = []
ydata_times = []

for size in SIZES:
    print(f'\nBenchmarking {size:,} records...')
    df = generate_test_data(size)
    
    # ParseIQ
    t, res = benchmark_parseiq(df)
    parseiq_times.append(t)
    print(f'  ParseIQ: {t:.2f}s (quality: {res.overall_quality_score:.1f})')
    
    # ydata-profiling
    t_ydata = benchmark_ydata(df)
    ydata_times.append(t_ydata)
    if t_ydata is not None:
        print(f'  ydata-profiling: {t_ydata:.2f}s')
    else:
        print(f'  ydata-profiling: not installed')

print('\nBenchmarks complete.')

In [ ]:
# Plot execution times
if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    ax.plot(SIZES, parseiq_times, 'o-', label='ParseIQ (local mode)', linewidth=2, markersize=8, color='#e94560')
    
    if any(t is not None for t in ydata_times):
        valid = [(s, t) for s, t in zip(SIZES, ydata_times) if t is not None]
        if valid:
            ax.plot([s for s, _ in valid], [t for _, t in valid], 's-', 
                    label='ydata-profiling (minimal)', linewidth=2, markersize=8, color='#0f3460')
    
    ax.set_xlabel('Number of Records', fontsize=12)
    ax.set_ylabel('Execution Time (seconds)', fontsize=12)
    ax.set_title('Execution Time vs Dataset Size', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.set_xscale('log')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('execution_time_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Chart saved: execution_time_comparison.png')
else:
    # Text-based table
    print('\nExecution Time Results:')
    print(f'{"Records":>10} | {"ParseIQ (s)":>12} | {"ydata (s)":>10}')
    print('-' * 40)
    for i, size in enumerate(SIZES):
        yd = f'{ydata_times[i]:.2f}' if ydata_times[i] is not None else 'N/A'
        print(f'{size:>10,} | {parseiq_times[i]:>12.2f} | {yd:>10}')

## 2. Anomaly Detection Accuracy

Measure precision, recall, and F1-score of ParseIQ's anomaly detectors against known ground truth.

In [ ]:
def create_ground_truth_dataset():
    """Create a dataset with known anomalies for accuracy measurement."""
    rng = np.random.default_rng(42)
    n = 100
    
    data = {
        # Column with HIGH_NULL_RATE (40% null -> should trigger)
        'high_null_col': [None if rng.random() < 0.4 else f'val_{i}' for i in range(n)],
        
        # Column with LOW_UNIQUENESS (only 3 values in 100 rows -> should trigger)
        'low_unique_col': rng.choice(['A', 'B', 'C'], size=n).tolist(),
        
        # Column with NUMERIC_OUTLIERS (inject Z>3 outlier)
        'outlier_col': np.concatenate([rng.normal(50, 5, n-2), [200, -100]]).tolist(),
        
        # Column with NEGATIVE_VALUES
        'negative_col': np.concatenate([rng.integers(1, 100, n-3), [-50, -20, -10]]).tolist(),
        
        # Column with MIXED_DATA_TYPES
        'mixed_col': [str(i) if i % 3 == 0 else i for i in range(n)],
        
        # Clean column (no anomalies expected)
        'clean_int': rng.integers(1, 1000, size=n).tolist(),
        
        # Clean string column
        'clean_str': [f'item_{i:04d}' for i in range(n)],
        
        # Column with FUTURE_DATE
        'date_col': ['2025-01-15'] * (n-2) + ['2099-12-31', '2098-06-15'],
    }
    
    # Ground truth: which columns should have which anomalies
    ground_truth = {
        'high_null_col': ['HIGH_NULL_RATE'],
        'low_unique_col': ['LOW_UNIQUENESS'],
        'outlier_col': ['NUMERIC_OUTLIERS_DETECTED'],
        'negative_col': ['NEGATIVE_VALUES_DETECTED'],
        'mixed_col': ['MIXED_DATA_TYPES'],
        'clean_int': [],
        'clean_str': [],
        'date_col': ['FUTURE_DATE_DETECTED'],
    }
    
    return data, ground_truth


data, ground_truth = create_ground_truth_dataset()
print(f'Ground truth dataset: {len(data["clean_int"])} records, {len(data)} columns')
print(f'Expected anomalies: {sum(len(v) for v in ground_truth.values())}')
print(f'Expected clean columns: {sum(1 for v in ground_truth.values() if len(v) == 0)}')

In [ ]:
# Run ParseIQ on the ground truth dataset
os.makedirs('accuracy_test', exist_ok=True)
temp_path = 'accuracy_test/ground_truth.json'
with open(temp_path, 'w') as f:
    json.dump(data, f, default=str)

result = Pipeline(temp_path).run(llm=False, output_dir='accuracy_test', force=True)

# Load raw metadata to extract per-column anomalies
raw_meta_path = os.path.join('accuracy_test', 'raw_metadata.json')
with open(raw_meta_path, 'r') as f:
    raw_meta = json.load(f)

# Extract detected anomalies per column
detected = {}
for table_name, table_data in raw_meta.get('tables', {}).items():
    attrs = table_data.get('table_metadata', {}).get('attributes', {})
    for col_name, col_data in attrs.items():
        anomalies = col_data.get('anomaly_types', [])
        if isinstance(anomalies, str):
            anomalies = [a.strip() for a in anomalies.split(',') if a.strip()]
        detected[col_name] = anomalies

print('\nDetected anomalies per column:')
for col, flags in sorted(detected.items()):
    gt = ground_truth.get(col, [])
    status = 'MATCH' if set(flags) >= set(gt) and (not flags or gt) else ('MISS' if gt and not set(flags) >= set(gt) else ('FP' if flags and not gt else 'OK'))
    print(f'  {col:20s} | detected: {flags or "(none)":<45s} | expected: {gt or "(none)":<35s} | {status}')

os.remove(temp_path)

In [ ]:
# Calculate precision, recall, F1
true_positives = 0
false_positives = 0
false_negatives = 0
true_negatives = 0

all_anomaly_types = set()
for flags in list(ground_truth.values()) + list(detected.values()):
    all_anomaly_types.update(flags)

for col in ground_truth:
    expected = set(ground_truth[col])
    found = set(detected.get(col, []))
    
    for atype in all_anomaly_types:
        if atype in expected and atype in found:
            true_positives += 1
        elif atype not in expected and atype in found:
            false_positives += 1
        elif atype in expected and atype not in found:
            false_negatives += 1
        else:
            true_negatives += 1

precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f'\n=== Anomaly Detection Accuracy ===')
print(f'True Positives:  {true_positives}')
print(f'False Positives: {false_positives}')
print(f'False Negatives: {false_negatives}')
print(f'True Negatives:  {true_negatives}')
print(f'\nPrecision: {precision:.3f}')
print(f'Recall:    {recall:.3f}')
print(f'F1 Score:  {f1:.3f}')

In [ ]:
# Visualise accuracy metrics
if HAS_MPL:
    metrics = ['Precision', 'Recall', 'F1 Score']
    values = [precision, recall, f1]
    colors = ['#e94560', '#0f3460', '#533483']
    
    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(metrics, values, color=colors, width=0.5, edgecolor='white', linewidth=1.5)
    
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', va='bottom', fontsize=13, fontweight='bold')
    
    ax.set_ylim(0, 1.15)
    ax.set_ylabel('Score', fontsize=12)
    ax.set_title('ParseIQ Anomaly Detection Accuracy', fontsize=14, fontweight='bold')
    ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('anomaly_detection_accuracy.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Chart saved: anomaly_detection_accuracy.png')

## 3. Quality Score Analysis

Analyse how ParseIQ's quality scores distribute across tables and correlate with anomaly counts.

In [ ]:
# Run ParseIQ on the stress test dataset (if available)
stress_path = os.path.join('..', 'input', 'stress_test_data.json')
enterprise_path = os.path.join('..', 'input', 'input_data.json')

datasets = {}
for name, path in [('stress_test', stress_path), ('enterprise', enterprise_path)]:
    if os.path.exists(path):
        print(f'Running ParseIQ on {name}...')
        out_dir = f'quality_analysis_{name}'
        os.makedirs(out_dir, exist_ok=True)
        result = Pipeline(path).run(llm=False, output_dir=out_dir, force=True)
        datasets[name] = result
        print(f'  Tables: {len(result.tables)}')
        print(f'  Overall quality: {result.overall_quality_score:.1f}')
        print(f'  Total anomalies: {result.total_anomalies}')
        print(f'  Per-table scores: {result.quality_scores}')
    else:
        print(f'{name}: file not found at {path}')

if not datasets:
    print('\nNo input files found. Run from the research/ directory inside the ParseIQ project.')

In [ ]:
# Plot quality score distribution
if HAS_MPL and datasets:
    fig, axes = plt.subplots(1, len(datasets), figsize=(7 * len(datasets), 6))
    if len(datasets) == 1:
        axes = [axes]
    
    for ax, (name, result) in zip(axes, datasets.items()):
        scores = result.quality_scores
        tables = list(scores.keys())
        values = list(scores.values())
        
        # Color by score
        colors = ['#22c55e' if v >= 80 else '#f59e0b' if v >= 50 else '#ef4444' for v in values]
        
        bars = ax.barh(tables, values, color=colors, edgecolor='white', linewidth=0.5)
        ax.set_xlim(0, 105)
        ax.set_xlabel('Quality Score (0-100)', fontsize=11)
        ax.set_title(f'{name} — Per-Table Quality', fontsize=13, fontweight='bold')
        ax.axvline(x=80, color='green', linestyle='--', alpha=0.3, label='Good threshold')
        ax.axvline(x=50, color='orange', linestyle='--', alpha=0.3, label='Warning threshold')
        
        for bar, val in zip(bars, values):
            ax.text(val + 1, bar.get_y() + bar.get_height() / 2,
                    f'{val:.0f}', va='center', fontsize=9)
        
        ax.legend(fontsize=9)
    
    plt.tight_layout()
    plt.savefig('quality_score_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Chart saved: quality_score_distribution.png')

## 4. Memory Usage Profiling

Measure peak memory consumption during analysis for different dataset sizes.

In [ ]:
# Memory profiling across dataset sizes
memory_sizes = [1_000, 5_000, 10_000, 50_000]
memory_results = []

for size in memory_sizes:
    print(f'Memory profiling {size:,} records...')
    df = generate_test_data(size)
    
    os.makedirs('memory_test', exist_ok=True)
    temp_path = 'memory_test/mem_data.json'
    records = df.astype(object).where(df.notna(), None).to_dict(orient='records')
    with open(temp_path, 'w') as f:
        json.dump(records, f, default=str)
    
    data_size_mb = os.path.getsize(temp_path) / (1024 * 1024)
    
    tracemalloc.start()
    result = Pipeline(temp_path).run(llm=False, output_dir='memory_test', force=True)
    _, peak_mb = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    peak_mb = peak_mb / (1024 * 1024)
    
    memory_results.append({
        'records': size,
        'data_size_mb': data_size_mb,
        'peak_memory_mb': peak_mb,
        'ratio': peak_mb / data_size_mb if data_size_mb > 0 else 0,
    })
    print(f'  Data: {data_size_mb:.1f} MB, Peak memory: {peak_mb:.1f} MB, Ratio: {peak_mb/data_size_mb:.1f}x')
    
    os.remove(temp_path)

mem_df = pd.DataFrame(memory_results)
print('\n', mem_df.to_string(index=False))

In [ ]:
# Plot memory usage
if HAS_MPL and memory_results:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    records = [r['records'] for r in memory_results]
    peaks = [r['peak_memory_mb'] for r in memory_results]
    data_sizes = [r['data_size_mb'] for r in memory_results]
    ratios = [r['ratio'] for r in memory_results]
    
    # Absolute memory
    ax1.plot(records, peaks, 'o-', color='#e94560', linewidth=2, markersize=8, label='Peak Memory')
    ax1.plot(records, data_sizes, 's--', color='#0f3460', linewidth=2, markersize=8, label='Input Data Size')
    ax1.set_xlabel('Records', fontsize=11)
    ax1.set_ylabel('Size (MB)', fontsize=11)
    ax1.set_title('Memory Usage vs Dataset Size', fontsize=13, fontweight='bold')
    ax1.legend(fontsize=10)
    ax1.set_xscale('log')
    ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    
    # Memory ratio
    ax2.bar([f'{r:,}' for r in records], ratios, color='#533483', edgecolor='white')
    ax2.set_xlabel('Records', fontsize=11)
    ax2.set_ylabel('Peak Memory / Data Size', fontsize=11)
    ax2.set_title('Memory Overhead Ratio', fontsize=13, fontweight='bold')
    ax2.axhline(y=2, color='gray', linestyle='--', alpha=0.3, label='2x baseline')
    ax2.legend(fontsize=10)
    
    plt.tight_layout()
    plt.savefig('memory_usage.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Chart saved: memory_usage.png')

## 5. Nested JSON Flattening Performance

Measure flattening time vs nesting depth and number of records.

In [ ]:
def generate_nested_json(depth, records_per_level=10):
    """Generate JSON data with specified nesting depth."""
    def build_level(d, parent_idx=0):
        if d <= 0:
            return {'value': np.random.randint(0, 1000), 'label': f'leaf_{parent_idx}'}
        
        record = {
            'id': parent_idx,
            'name': f'level_{d}_item_{parent_idx}',
            'score': round(np.random.uniform(0, 100), 2),
        }
        
        # Add nested children
        record[f'children_L{d}'] = [
            build_level(d - 1, i) for i in range(records_per_level)
        ]
        
        return record
    
    return [build_level(depth, i) for i in range(records_per_level)]


depths = [1, 2, 3, 5, 7, 10]
flatten_times = []
table_counts = []

for depth in depths:
    print(f'Depth {depth}...')
    data = generate_nested_json(depth, records_per_level=5)
    
    os.makedirs('flatten_test', exist_ok=True)
    temp_path = 'flatten_test/nested.json'
    with open(temp_path, 'w') as f:
        json.dump(data, f, default=str)
    
    file_size_mb = os.path.getsize(temp_path) / (1024 * 1024)
    
    start = time.perf_counter()
    result = Pipeline(temp_path).run(llm=False, output_dir='flatten_test', force=True)
    elapsed = time.perf_counter() - start
    
    n_tables = len(result.tables)
    flatten_times.append(elapsed)
    table_counts.append(n_tables)
    
    print(f'  Depth: {depth}, File: {file_size_mb:.2f} MB, Tables: {n_tables}, Time: {elapsed:.2f}s')
    os.remove(temp_path)

print('\nFlattening benchmarks complete.')

In [ ]:
# Plot flattening performance
if HAS_MPL:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    ax1.plot(depths, flatten_times, 'o-', color='#e94560', linewidth=2, markersize=8)
    ax1.set_xlabel('Nesting Depth', fontsize=11)
    ax1.set_ylabel('Total Analysis Time (s)', fontsize=11)
    ax1.set_title('Analysis Time vs Nesting Depth', fontsize=13, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    
    ax2.bar([str(d) for d in depths], table_counts, color='#0f3460', edgecolor='white')
    ax2.set_xlabel('Nesting Depth', fontsize=11)
    ax2.set_ylabel('Tables Extracted', fontsize=11)
    ax2.set_title('Tables Extracted vs Nesting Depth', fontsize=13, fontweight='bold')
    
    for i, (d, tc) in enumerate(zip(depths, table_counts)):
        ax2.text(i, tc + 0.3, str(tc), ha='center', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('flattening_performance.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Chart saved: flattening_performance.png')

## 6. Feature Comparison Matrix

Comprehensive comparison with existing data quality tools.

In [ ]:
# Feature comparison data
comparison = pd.DataFrame({
    'Feature': [
        'Zero configuration',
        'Nested JSON support',
        'Multi-format (JSON/CSV/XML/Excel)',
        'Automated anomaly detection',
        'Schema polymorphism detection',
        'LLM integration (multi-provider)',
        'Quality scoring (0-100)',
        'Cross-table analysis',
        'Incremental processing',
        'Web UI',
        'BYOK (data privacy)',
        'Local/offline mode',
        'Excel report output',
        'CI/CD quality gate',
        'Custom rules engine',
        'Alert callbacks',
    ],
    'ParseIQ': ['Yes', 'Arbitrary depth', 'Yes', '11 types', 'Yes', '7 providers', 
                'Per-attr + per-table', 'Range + constraint', 'Hash-based',
                'Yes (React+FastAPI)', 'Yes', 'Yes', '30-col Meta sheet',
                '--fail-under', 'YAML/JSON sidecar', 'Slack/email'],
    'ydata-profiling': ['Yes', 'No', 'CSV/DataFrame', 'Warnings', 'No', 'No',
                        'Correlations', 'No', 'No', 'HTML report', 'N/A', 'Yes',
                        'No', 'No', 'No', 'No'],
    'Great Expectations': ['No (YAML)', 'No', 'Via connectors', 'User-defined', 'No', 'No',
                           'Pass/Fail', 'Via suites', 'Checkpoint', 'Data Docs (static)',
                           'N/A', 'Yes', 'No', 'Yes', 'Expectation suites', 'Actions'],
    'pandas-profiling': ['Yes', 'No', 'CSV/DataFrame', 'Warnings', 'No', 'No',
                         'Limited', 'No', 'No', 'HTML report', 'N/A', 'Yes',
                         'No', 'No', 'No', 'No'],
})

print(comparison.to_string(index=False))

In [ ]:
# Capability scoring for radar chart
categories = [
    'Ease of Use',
    'Format Support',
    'Anomaly Detection',
    'AI Integration',
    'Reporting',
    'Scalability',
    'Data Privacy',
    'Extensibility',
]

# Scores 1-5
scores = {
    'ParseIQ':           [5, 5, 5, 5, 5, 4, 5, 4],
    'ydata-profiling':   [5, 2, 2, 1, 3, 3, 4, 2],
    'Great Expectations': [2, 4, 3, 1, 3, 5, 4, 5],
    'pandas-profiling':  [5, 2, 2, 1, 3, 2, 4, 1],
}

if HAS_MPL:
    # Radar chart
    angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
    angles += angles[:1]  # close the polygon
    
    fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
    
    colors_radar = ['#e94560', '#0f3460', '#f59e0b', '#22c55e']
    
    for (name, vals), color in zip(scores.items(), colors_radar):
        vals_closed = vals + vals[:1]
        ax.plot(angles, vals_closed, 'o-', linewidth=2, label=name, color=color, markersize=6)
        ax.fill(angles, vals_closed, alpha=0.1, color=color)
    
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=10)
    ax.set_ylim(0, 5.5)
    ax.set_yticks([1, 2, 3, 4, 5])
    ax.set_yticklabels(['1', '2', '3', '4', '5'], fontsize=8, color='gray')
    ax.set_title('Tool Capability Comparison', fontsize=14, fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=10)
    
    plt.tight_layout()
    plt.savefig('capability_radar.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Chart saved: capability_radar.png')
else:
    print('\nCapability Scores (1-5):')
    cap_df = pd.DataFrame(scores, index=categories)
    print(cap_df.to_string())

## 7. Summary Statistics

Aggregate all benchmark results into a summary table suitable for a research paper.

In [ ]:
print('=' * 70)
print('PARSEIQ PERFORMANCE SUMMARY')
print('=' * 70)

print(f'\nVersion: {__version__}')
print(f'Python: {sys.version.split()[0]}')

print(f'\n--- Execution Time (local mode, 10 columns) ---')
for size, t in zip(SIZES, parseiq_times):
    print(f'  {size:>8,} records: {t:>6.2f}s')

print(f'\n--- Anomaly Detection Accuracy ---')
print(f'  Precision: {precision:.3f}')
print(f'  Recall:    {recall:.3f}')
print(f'  F1 Score:  {f1:.3f}')

if memory_results:
    print(f'\n--- Memory Usage ---')
    for r in memory_results:
        print(f'  {r["records"]:>8,} records: {r["peak_memory_mb"]:.1f} MB peak ({r["ratio"]:.1f}x data size)')

print(f'\n--- Nested JSON Flattening ---')
for d, t, tc in zip(depths, flatten_times, table_counts):
    print(f'  Depth {d:>2}: {t:>6.2f}s, {tc} tables extracted')

print(f'\n--- Test Suite ---')
print(f'  Total tests: 590+')
print(f'  Test domains: 10 (e-commerce, HR, hospital, university, supply chain, banking, social media, IoT, insurance, conglomerate)')

print(f'\n--- Anomaly Detectors ---')
print(f'  Column-level: 8 (always active)')
print(f'  Cross-table: 2 (automatic)')
print(f'  Rule-based: 2 (YAML/JSON sidecar)')
print(f'  Total: 11 + 1 informational (TYPE_CONDITIONAL_FIELD)')

print('\n' + '=' * 70)

## 8. Cleanup

Remove temporary benchmark files.

In [ ]:
import shutil

for d in ['benchmark_output', 'accuracy_test', 'memory_test', 'flatten_test',
          'quality_analysis_stress_test', 'quality_analysis_enterprise']:
    if os.path.exists(d):
        shutil.rmtree(d)
        print(f'Removed: {d}/')

print('Cleanup complete.')